# Домашнее задание: Градиентный бустинг

### Цели работы
1. Понять принцип градиентного спуска в функциональном пространстве
2. Реализовать базовый градиентный бустинг для регрессии
3. Исследовать влияние learning rate и количества деревьев
4. Сравнить с sklearn.GradientBoostingRegressor

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor

# Фиксация генератора случайных чисел для воспроизводимости
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)

## Теория: Градиентный бустинг

### Идея

**Градиентный бустинг** — это ансамблевый метод, который строит модели последовательно. Каждая новая модель исправляет ошибки предыдущих.

В отличие от **Random Forest**, где деревья строятся **параллельно** и независимо, в бустинге каждое дерево учится на **ошибках** предыдущих.

### Градиентный спуск в функциональном пространстве

Обычный градиентный спуск минимизирует функцию потерь, изменяя **параметры** (веса) модели:

$$w_{t+1} = w_t - \eta \cdot \nabla L(w_t)$$

В градиентном бустинге мы минимизируем функцию потерь, добавляя **новые функции** (деревья) в ансамбль:

$$F_{m}(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

где:
- $F_m(x)$ — предсказание ансамбля после $m$ итераций
- $\eta$ (learning rate) — скорость обучения (шаг)
- $h_m(x)$ — новое дерево, обученное на **антиградиенте** функции потерь

### Почему для MSE остатки — это просто $y - F(x)$?

Это ключевой момент! Давайте вычислим градиент MSE.

Функция потерь MSE для одного объекта:

$$L(y, F(x)) = (y - F(x))^2$$

Градиент (производная) по предсказанию $F(x)$:

$$\frac{\partial L}{\partial F(x)} = -2(y - F(x))$$

**Антиградиент** (направление наискорейшего убывания):

$$-\frac{\partial L}{\partial F(x)} = 2(y - F(x)) \propto y - F(x)$$

Поэтому мы обучаем каждое новое дерево на **остатках** $r_m = y - F_{m-1}(x)$ — это и есть антиградиент MSE!

### Алгоритм для регрессии (MSE как функция потерь)

1. **Инициализация**: $F_0(x) = \bar{y}$ (среднее значение целевой переменной)

2. **Для каждого $m = 1, \ldots, M$**:
   - Вычислить **остатки** (антиградиент MSE): $r_m = y - F_{m-1}(x)$
   - Обучить дерево $h_m(x)$ на остатках $r_m$ (дерево аппроксимирует антиградиент)
   - Обновить модель: $F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$

3. **Предсказание**: $\hat{y} = F_M(x)$

### Геометрическая интерпретация

Каждое новое дерево делает шаг в направлении, которое уменьшает ошибку быстрее всего:
- Остатки показывают, **где** и **насколько** мы ошибаемся
- Дерево учится предсказывать эти остатки
- Добавляя предсказание дерева, мы исправляем ошибку

### Ключевые гиперпараметры

| Параметр | Описание |
|----------|----------|
| `n_estimators` | Количество деревьев (итераций бустинга) |
| `learning_rate` | Шаг градиентного спуска (обычно 0.01–0.1) |
| `max_depth` | Глубина каждого дерева (обычно 3–8 для слабых learners) |

### Взаимосвязь learning rate и n_estimators

- **Маленький learning_rate** требуется больше деревьев, но даёт более устойчивую модель
- **Большой learning_rate** сходится быстрее, но может «перепрыгнуть» оптимум

Бустинг менее склонен к переобучению с ростом `n_estimators` (по сравнению с Random Forest), если использовать достаточно маленький `learning_rate`.

## Ваша реализация Градиентного Бустинга

### Задача: Реализовать градиентный спуск в функциональном пространстве

**Важно:** Используем `DecisionTreeRegressor` из sklearn как базовый learner.

Вам нужно реализовать только сам алгоритм градиентного бустинга:

In [ ]:
class MyGradientBoostingRegressor:
    """
    Ваша реализация Градиентного Бустинга для регрессии.

    Использует DecisionTreeRegressor из sklearn как базовый learner.

    Параметры:
    -----------
    n_estimators : int
        Количество деревьев (итераций бустинга)
    learning_rate : float
        Скорость обучения (шаг градиентного спуска)
    max_depth : int
        Максимальная глубина каждого дерева (слабые learners)
    random_state : int
        Фиксация случайности
    """

    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3,
                 random_state=42):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.random_state = random_state
        self.trees = []
        self.initial_pred = None

    def fit(self, X, y):
        """
        Обучение градиентного бустинга.

        Алгоритм:
        1. Инициализировать начальное предсказание как среднее y
        2. Для каждой итерации:
           a. Вычислить остатки (residuals) = y - текущие_предсказания
           b. Обучить DecisionTreeRegressor на остатках
           c. Добавить дерево в ансамбль
           d. Обновить предсказания
        """
        #TODO: Реализовать алгоритм
        # Подсказка: используйте DecisionTreeRegressor из sklearn
        pass

    def predict(self, X):
        """
        Предсказание целевых значений.

        1. Начать с начального предсказания
        2. Для каждого дерева добавить learning_rate * tree.predict(X)
        3. Вернуть итоговое предсказание
        """
        #TODO: Реализовать предсказание
        pass

---

## Задание 0: Самопроверка

Тест на простых данных (y = sin(x) + noise). Это поможет вам убедиться, что ваша реализация работает корректно.

In [ ]:
# Генерация простых 1D данных для визуализации
np.random.seed(RANDOM_STATE)
X_simple = np.linspace(0, 4 * np.pi, 100).reshape(-1, 1)
y_simple = np.sin(X_simple.ravel()) + np.random.normal(0, 0.2, len(X_simple))

# Разделение на train/test
X_train_simple, X_test_simple, y_train_simple, y_test_simple = train_test_split(
    X_simple, y_simple, test_size=0.3, random_state=RANDOM_STATE
)

print(f"Train: {X_train_simple.shape}, Test: {X_test_simple.shape}")

In [ ]:
#TODO: Создайте и обучите вашу модель
# Используйте параметры: n_estimators=50, learning_rate=0.1, max_depth=3
my_gb = MyGradientBoostingRegressor(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    random_state=RANDOM_STATE
)

#TODO: Обучите модель
# my_gb.fit(...)

#TODO: Сделайте предсказания
# y_pred_train = my_gb.predict(...)
# y_pred_test = my_gb.predict(...)

In [ ]:
# Визуализация результата
plt.figure(figsize=(12, 5))

# Train
plt.subplot(1, 2, 1)
plt.scatter(X_train_simple, y_train_simple, alpha=0.5, label='Data')
plt.scatter(X_train_simple, y_pred_train, color='red', alpha=0.5, label='Prediction')
plt.title('Train: sin(x) + noise')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()

# Test
plt.subplot(1, 2, 2)
plt.scatter(X_test_simple, y_test_simple, alpha=0.5, label='Data')
plt.scatter(X_test_simple, y_pred_test, color='red', alpha=0.5, label='Prediction')
plt.title('Test: sin(x) + noise')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()

plt.tight_layout()
plt.show()

print(f"Train MSE: {mean_squared_error(y_train_simple, y_pred_train):.4f}")
print(f"Test MSE:  {mean_squared_error(y_test_simple, y_pred_test):.4f}")
print(f"Train R2:  {r2_score(y_train_simple, y_pred_train):.4f}")
print(f"Test R2:   {r2_score(y_test_simple, y_pred_test):.4f}")

In [ ]:
# ТЕСТЫ ДЛЯ ЗАДАНИЯ 0
#assert len(my_gb.trees) == 50, "Должно быть обучено 50 деревьев"
#assert my_gb.initial_pred is not None, "Начальное предсказание не инициализировано"
#assert r2_score(y_test_simple, y_pred_test) > 0.5, "R2 должен быть > 0.5 на этих данных"
print("Разкомментируйте тесты после реализации класса MyGradientBoostingRegressor")

---

## Задание 1: Влияние Learning Rate

Цель: понять как learning rate влияет на сходимость и качество модели.

In [ ]:
# Генерация синтетических данных
X, y = make_regression(
    n_samples=1000,
    n_features=10,
    noise=20,
    random_state=RANDOM_STATE
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Различные значения learning rate
learning_rates = [0.001, 0.01, 0.1, 0.5, 1.0]
train_scores = []
test_scores = []

#TODO: Для каждого learning rate из списка:
# 1. Обучите MyGradientBoostingRegressor с n_estimators=100, max_depth=3
# 2. Вычислите R2 на train и test
# 3. Добавьте результаты в списки train_scores и test_scores

for lr in learning_rates:
    # ВАШ КОД ЗДЕСЬ
    pass

# --- Визуализация ---
plt.figure(figsize=(10, 6))
plt.semilogx(learning_rates, train_scores, marker='o', label='Train R²')
plt.semilogx(learning_rates, test_scores, marker='s', label='Test R²')
plt.xlabel('Learning Rate (log scale)')
plt.ylabel('R² Score')
plt.title('Влияние Learning Rate на качество модели')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

for lr, train_r2, test_r2 in zip(learning_rates, train_scores, test_scores):
    print(f"LR={lr:.4f}: Train R²={train_r2:.4f}, Test R²={test_r2:.4f}")

In [ ]:
# ТЕСТЫ ДЛЯ ЗАДАНИЯ 1
assert len(train_scores) == len(learning_rates), "train_scores должен иметь такую же длину как learning_rates"
assert len(test_scores) == len(learning_rates), "test_scores должен иметь такую же длину как learning_rates"
assert max(test_scores) > 0.5, "Максимальный R2 на test должен быть > 0.5"
print("✓ Тесты пройдены!")

---

## Задание 2: Количество деревьев и переобучение

Цель: наблюдать как больше деревьев влияет на модель. В отличие от Random Forest, градиентный бустинг может начать переобучиваться при слишком большом количестве деревьев.

In [ ]:
# Используем те же данные
n_estimators_list = [10, 50, 100, 200, 500]
train_mses = []
test_mses = []

#TODO: Для каждого значения n_estimators из списка:
# 1. Обучите MyGradientBoostingRegressor с learning_rate=0.1, max_depth=3
# 2. Вычислите MSE на train и test
# 3. Добавьте результаты в списки train_mses и test_mses

for n_est in n_estimators_list:
    # ВАШ КОД ЗДЕСЬ
    pass

# --- Визуализация ---
plt.figure(figsize=(10, 6))
plt.plot(n_estimators_list, train_mses, marker='o', label='Train MSE')
plt.plot(n_estimators_list, test_mses, marker='s', label='Test MSE')
plt.xlabel('Количество деревьев (n_estimators)')
plt.ylabel('MSE')
plt.title('MSE vs Количество деревьев')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

for n_est, train_mse, test_mse in zip(n_estimators_list, train_mses, test_mses):
    print(f"n_est={n_est:3d}: Train MSE={train_mse:.2f}, Test MSE={test_mse:.2f}")

In [ ]:
# ТЕСТЫ ДЛЯ ЗАДАНИЯ 2
assert len(train_mses) == len(n_estimators_list), "train_mses должен иметь такую же длину как n_estimators_list"
assert len(test_mses) == len(n_estimators_list), "test_mses должен иметь такую же длину как n_estimators_list"
assert train_mses[0] > train_mses[-1], "Train MSE должен уменьшаться с ростом деревьев"
print("✓ Тесты пройдены!")

---

## Задание 3: Визуализация итеративного улучшения

Цель: увидеть как модель улучшается с каждой итерацией. Наглядно покажем, как кусочно-линейная аппроксимация приближает плавную функцию.

In [ ]:
# Используем 1D данные для наглядности
# Генерируем более гладкую функцию
np.random.seed(RANDOM_STATE)
X_viz = np.linspace(0, 6, 200).reshape(-1, 1)
y_viz = np.sin(X_viz.ravel()) * 2 + np.random.normal(0, 0.3, len(X_viz))

X_train_viz, X_test_viz, y_train_viz, y_test_viz = train_test_split(
    X_viz, y_viz, test_size=0.3, random_state=RANDOM_STATE
)

print(f"Train: {X_train_viz.shape}, Test: {X_test_viz.shape}")

In [ ]:
#TODO: Обучите модель с n_estimators=100, learning_rate=0.1, max_depth=3
# Сохраните обученную модель в переменную gb_viz

# ВАШ КОД ЗДЕСЬ
gb_viz = None

In [ ]:
# Визуализация предсказаний после разного числа деревьев
estimators_to_show = [1, 5, 10, 25, 50, 100]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

# Для красивой визуализации сортируем данные по x
sort_idx = np.argsort(X_test_viz.ravel())
X_sorted = X_test_viz[sort_idx]
y_sorted = y_test_viz[sort_idx]

for idx, n_est in enumerate(estimators_to_show):
    ax = axes[idx]

    #TODO: Получите предсказание используя только первые n_est деревьев
    # Подсказка: вам нужно будет использовать gb_viz.initial_pred и первые n_est деревьев
    # из gb_viz.trees

    # ВАШ КОД ЗДЕСЬ
    # y_partial = ...

    # Если ещё не реализовали, используйте заглушку
    if gb_viz is None or len(gb_viz.trees) == 0:
        y_partial = y_sorted
    else:
        #TODO: реализуйте вычисление y_partial для первых n_est деревьев
        y_partial = y_sorted

    # Визуализация
    ax.scatter(X_test_viz, y_test_viz, alpha=0.3, label="Data")
    ax.plot(X_sorted, y_partial, color="red", linewidth=2, label=f"After {n_est} trees")
    ax.set_title(f"После {n_est} деревьев")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Задание 4: Сравнение со sklearn

Цель: сравнить вашу реализацию с sklearn.GradientBoostingRegressor на реальном датасете.

In [ ]:
# Загрузка California Housing dataset
data = fetch_california_housing()
X_cal, y_cal = data.data, data.target
feature_names = data.feature_names

print(f"Features: {feature_names}")
print(f"Shape: {X_cal.shape}")

In [ ]:
# Разделение данных
X_train_cal, X_test_cal, y_train_cal, y_test_cal = train_test_split(
    X_cal, y_cal, test_size=0.3, random_state=RANDOM_STATE
)

print(f"Train: {X_train_cal.shape}, Test: {X_test_cal.shape}")

In [ ]:
#TODO: Обучите вашу реализацию
# Параметры: n_estimators=100, learning_rate=0.1, max_depth=3

# ВАШ КОД ЗДЕСЬ
# my_gb_cal = ...
# my_gb_cal.fit(...)
# y_pred_my = my_gb_cal.predict(...)
# my_r2 = r2_score(...)

In [ ]:
#TODO: Обучите sklearn GradientBoostingRegressor с теми же параметрами

# ВАШ КОД ЗДЕСЬ
# sk_gb = ...
# sk_gb.fit(...)
# y_pred_sk = sk_gb.predict(...)
# sk_r2 = r2_score(...)

In [ ]:
# Сравнение результатов
print("Сравнение R² Score:")
print(f"  Ваша реализация: {my_r2:.4f}")
print(f"  Sklearn:          {sk_r2:.4f}")
print(f"  Разница:          {abs(my_r2 - sk_r2):.4f}")

print("\nСравнение MSE:")
print(f"  Ваша реализация: {mean_squared_error(y_test_cal, y_pred_my):.4f}")
print(f"  Sklearn:          {mean_squared_error(y_test_cal, y_pred_sk):.4f}")

In [ ]:
# Feature importance comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Ваша реализация - простая версия (считаем по частоте использования признаков)
if hasattr(my_gb_cal, "trees") and len(my_gb_cal.trees) > 0:
    # Упрощенная важность признаков: среднее снижение impurity
    importances = np.zeros(X_cal.shape[1])
    for tree in my_gb_cal.trees:
        if hasattr(tree, "feature_importances_"):
            importances += tree.feature_importances_
    importances /= len(my_gb_cal.trees)

    ax1.barh(feature_names, importances)
    ax1.set_xlabel("Importance")
    ax1.set_title("Ваша реализация")
    ax1.grid(True, alpha=0.3)

# Sklearn
ax2.barh(feature_names, sk_gb.feature_importances_)
ax2.set_xlabel("Importance")
ax2.set_title("Sklearn GradientBoostingRegressor")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ТЕСТЫ ДЛЯ ЗАДАНИЯ 4
assert my_r2 > 0.5, "R2 вашей реализации должен быть > 0.5"
assert sk_r2 > 0.5, "R2 sklearn должен быть > 0.5"
assert abs(my_r2 - sk_r2) < 0.15, "Разница между реализациями должна быть < 0.15"
print("✓ Тесты пройдены!")

---

## Задание 5: Анализ остатков

Цель: понять как GB уменьшает остатки итеративно. Посмотрим на распределение остатков после разного числа итераций.

In [ ]:
# Используем California Housing данные
#TODO: Обучите модель с n_estimators=200

# ВАШ КОД ЗДЕСЬ
# gb_residuals = ...
# gb_residuals.fit(...)

print("Разкомментируйте и реализуйте обучение модели")

In [ ]:
#TODO: Вычислите остатки на train после разного числа итераций
# Итерации: [1, 10, 50, 100, 200]

iterations = [1, 10, 50, 100, 200]
residuals_dict = {}

# ВАШ КОД ЗДЕСЬ
# Для каждой итерации:
# 1. Получите предсказание используя только первые n деревьев
# 2. Вычислите остатки: y_train - prediction
# 3. Сохраните в residuals_dict[n] = residuals

print("Разкомментируйте и реализуйте вычисление остатков")

In [ ]:
# Визуализация остатков
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

# Начальные остатки (до обучения - просто разность y и среднего)
if "gb_residuals" in locals() and hasattr(gb_residuals, "initial_pred"):
    initial_residuals = y_train_cal - gb_residuals.initial_pred

    axes[0].hist(initial_residuals, bins=30, alpha=0.7, edgecolor="black")
    axes[0].set_title("Начальные остатки (среднее)")
    axes[0].set_xlabel("Значение остатка")
    axes[0].set_ylabel("Частота")
    axes[0].grid(True, alpha=0.3)

    for idx, n_est in enumerate(iterations[:5], 1):
        if n_est in residuals_dict:
            axes[idx].hist(residuals_dict[n_est], bins=30, alpha=0.7, edgecolor="black")
            axes[idx].set_title(f"После {n_est} деревьев")
            axes[idx].set_xlabel("Значение остатка")
            axes[idx].set_ylabel("Частота")
            axes[idx].grid(True, alpha=0.3)

            # Добавляем статистику
            std = np.std(residuals_dict[n_est])
            mean = np.mean(residuals_dict[n_est])
            axes[idx].text(
                0.05,
                0.95,
                f"μ={mean:.3f}\nσ={std:.3f}",
                transform=axes[idx].transAxes,
                verticalalignment="top",
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
            )
else:
    for ax in axes:
        ax.text(0.5, 0.5, "Реализуйте обучение модели", ha="center", va="center", transform=ax.transAxes)
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()

---

## Итоги

Вы реализовали градиентный бустинг с нуля! Поздравляем!

### Что вы узнали:
1. **Идея бустинга**: последовательное исправление ошибок
2. **Градиентный спуск в функциональном пространстве**: добавляем новые функции вместо изменения параметров
3. **Влияние гиперпараметров**:
   - Learning rate контролирует размер шага
   - N_estimators определяет количество итераций
4. **Отличия от Random Forest**:
   - Последовательное vs параллельное построение
   - Бустинг может переобучиваться при большом количестве деревьев
   - Бустинг обычно даёт лучшее качество, но более чувствителен к гиперпараметрам

### Что дальше?

В sklearn `GradientBoostingRegressor` есть дополнительные улучшения:
- **Подвыборка (subsample)**: stochastic gradient boosting для регуляризации
- **Инициализация**: можно начинать не только со среднего, но и с другой модели
- **Различные функции потерь**: не только MSE, но и MAE, Huber loss и др.
- **Пост-прuning**: ограничение на число листьев (max_leaf_nodes)